# Comprehensive Quantization Training Guide

This notebook provides a comprehensive guide to quantization methods in LLaMA-Factory, covering:

1. **Quantization Fundamentals**: Understanding model quantization
2. **AWQ Training**: Activation-aware Weight Quantization
3. **GPTQ Training**: General Purpose Transformer Quantization
4. **AQLM Training**: Activation-aware Quantization for Language Models
5. **OTFQ Training**: On-the-fly Quantization
6. **Quantization Evaluation**: Performance benchmarking
7. **Advanced Techniques**: Mixed precision and optimization
8. **Best Practices**: Deployment and production considerations

## Table of Contents

- [Setup and Installation](#setup-and-installation)
- [Quantization Fundamentals](#quantization-fundamentals)
- [AWQ Training](#awq-training)
- [GPTQ Training](#gptq-training)
- [AQLM Training](#aqlm-training)
- [OTFQ Training](#otfq-training)
- [Evaluation](#evaluation)
- [Advanced Techniques](#advanced-techniques)
- [Best Practices](#best-practices)


## Setup and Installation

First, let's install the required dependencies for quantization training.


In [ ]:
# Install quantization dependencies
%pip install -r requirements.txt
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install transformers[torch] datasets accelerate bitsandbytes
%pip install autoawq optimum auto-gptq
%pip install matplotlib seaborn plotly pandas numpy
%pip install torch-fidelity  # For quantization quality assessment

# Import required libraries
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AwqConfig, GptqConfig
from peft import PeftModel
import bitsandbytes as bnb
import json
import os
import yaml
from typing import Dict, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


## Quantization Fundamentals

### What is Quantization?

Quantization reduces model precision from 32-bit or 16-bit floating point to lower precision (8-bit, 4-bit, or 3-bit) to reduce memory usage and improve inference speed while maintaining model quality.

### Types of Quantization

1. **Post-Training Quantization**: Quantize after training
2. **Quantization-Aware Training**: Train with quantization
3. **Dynamic Quantization**: Quantize during inference
4. **Static Quantization**: Pre-compute quantization parameters

### Quantization Precision

| Precision | Bits | Memory Reduction | Quality Loss |
|-----------|------|------------------|--------------|
| FP32 | 32 | 0% | None |
| FP16 | 16 | 50% | Minimal |
| INT8 | 8 | 75% | Low |
| INT4 | 4 | 87.5% | Moderate |
| INT3 | 3 | 90.6% | High |
